# Bayesian 기반 도로/차선 분석 과제 풀이

- 과제 노트북: `auto_class_2_BayesianClassifier.ipynb`
- 사용 sequence: **KITTI odometry sequence 09**
- 확인한 실제 데이터 경로: `D:\자율이동체\dataset`
- 실제 데이터 확인 결과: sequence 09 이미지 1591장, pose 1591개

데이터 구조는 다음을 가정한다.

```text
dataset/
  sequences/09/
    calib.txt
    image_0/000000.png ...
  poses/09.txt
```

노트북은 먼저 현재 폴더의 `dataset`을 찾고, 없으면 외장하드의 `D:\자율이동체\dataset`을 자동으로 사용한다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

CANDIDATE_DATA_ROOTS = [Path("dataset"), Path(r"D:\자율이동체\dataset")]
DATA_ROOT = next((p for p in CANDIDATE_DATA_ROOTS if (p / "sequences" / "09").exists()), CANDIDATE_DATA_ROOTS[0])
SEQ = "09"
SEQ_DIR = DATA_ROOT / "sequences" / SEQ
IMG_DIR = SEQ_DIR / "image_0"
CALIB_PATH = SEQ_DIR / "calib.txt"
POSE_PATH = DATA_ROOT / "poses" / f"{SEQ}.txt"
OUT_DIR = Path("outputs") / "sequence_09_solution"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT.resolve())
print("CALIB:", CALIB_PATH.exists(), "POSE:", POSE_PATH.exists(), "cv2:", HAS_CV2)

In [ ]:
def list_frames(img_dir=IMG_DIR):
    return sorted(img_dir.glob("*.png"))

def read_calib(path=CALIB_PATH):
    calib = {}
    if not path.exists():
        raise FileNotFoundError(f"calib.txt가 없습니다: {path}")
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        key, values = line.split(":", 1)
        vals = np.array([float(v) for v in values.split()], dtype=float)
        if key.startswith("P"):
            calib[key] = vals.reshape(3, 4)
        elif key == "Tr":
            calib[key] = vals.reshape(3, 4)
    return calib

def read_poses(path=POSE_PATH):
    if not path.exists():
        return np.empty((0, 3, 4), dtype=float)
    return np.stack([
        np.array([float(v) for v in line.split()], dtype=float).reshape(3, 4)
        for line in path.read_text().splitlines()
        if line.strip()
    ])

def project_points(P, points_xyz):
    points_xyz = np.asarray(points_xyz, dtype=float)
    points_h = np.c_[points_xyz, np.ones(len(points_xyz))]
    uvw = (P @ points_h.T).T
    valid = uvw[:, 2] > 1e-6
    uv = np.full((len(points_xyz), 2), np.nan)
    uv[valid] = uvw[valid, :2] / uvw[valid, 2:3]
    return uv, valid

frames = list_frames()
calib = read_calib()
poses = read_poses()
P0 = calib["P0"]
W, H = Image.open(frames[0]).size if frames else (1241, 376)
fx, fy, cx, cy = P0[0, 0], P0[1, 1], P0[0, 2], P0[1, 2]

print("frames:", len(frames), "poses:", len(poses), "image size:", (W, H))
print("P0 =\n", P0)
print(f"fx={fx:.4f}, fy={fy:.4f}, cx={cx:.4f}, cy={cy:.4f}")

## 문제 1. Projection Matrix 해석

KITTI odometry의 `P0`, `P1`, `P2`, `P3`는 rectified camera 좌표계의 3D 점을 각 카메라 이미지 평면으로 투영하는 `3 x 4` 행렬이다.

\[
\mathbf{x}_{img} \sim \mathbf{P}\mathbf{X}_{cam},\quad
\begin{bmatrix}u'\\v'\\w'\end{bmatrix}=
\begin{bmatrix}
f_x&0&c_x&p_{14}\\
0&f_y&c_y&p_{24}\\
0&0&1&p_{34}
\end{bmatrix}
\begin{bmatrix}X\\Y\\Z\\1\end{bmatrix}
\]

최종 픽셀 좌표는 다음처럼 homogeneous normalization으로 얻는다.

\[
u = \frac{u'}{w'},\qquad v = \frac{v'}{w'}
\]

- `f_x`, `f_y`: 초점거리이다. 값이 클수록 같은 3D 변화가 이미지에서 더 크게 나타난다.
- `c_x`, `c_y`: principal point이다. 카메라 광축이 이미지 평면과 만나는 기준점이다.
- `R`: world 또는 vehicle 좌표계를 camera 좌표계 방향으로 맞추는 회전 성분이다.
- `t`: 좌표계 원점 사이의 병진 이동이다. stereo camera에서는 baseline 차이가 projection matrix의 마지막 열에 반영된다.

일반적인 카메라 투영식은 다음과 같다.

\[
\lambda
\begin{bmatrix}u\\v\\1\end{bmatrix}
=
\mathbf{K}[\mathbf{R}|\mathbf{t}]
\begin{bmatrix}X_w\\Y_w\\Z_w\\1\end{bmatrix}
\]

sequence 09의 `P0`에서 추출한 intrinsic 값은 `fx=707.0912`, `fy=707.0912`, `cx=601.8873`, `cy=183.1104`이다.

## 문제 2. Projection Matrix를 이용한 3D → 2D 투영

아래 셀은 카메라 앞 ground plane 부근의 임의 3D 점들을 생성하고, `P0`를 이용하여 이미지 좌표로 투영한다.

In [ ]:
xs = np.linspace(-6, 6, 9)
zs = np.array([8, 12, 18, 25, 35, 50])
points = np.array([[x, 1.6, z] for z in zs for x in xs], dtype=float)
uv, valid = project_points(P0, points)
inside = valid & (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)

fig, ax = plt.subplots(figsize=(12, 4))
if frames:
    ax.imshow(Image.open(frames[0]).convert("RGB"))
else:
    ax.imshow(np.full((H, W, 3), 245, dtype=np.uint8))
sc = ax.scatter(uv[inside, 0], uv[inside, 1], c=points[inside, 2], cmap="viridis", s=35)
ax.scatter([cx], [cy], marker="x", c="red", s=80, label="principal point")
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_title("3D points projected by P0")
ax.legend()
plt.colorbar(sc, ax=ax, label="Z distance")
plt.show()

pd.DataFrame({
    "X": points[:, 0], "Y": points[:, 1], "Z": points[:, 2],
    "u": uv[:, 0], "v": uv[:, 1], "inside_image": inside
}).head(12)

투영 결과 해석: 가까운 점은 이미지 아래쪽과 바깥쪽으로 크게 퍼지고, 먼 점은 principal point `(c_x, c_y)` 방향으로 모인다. 따라서 실제 도로 위에서 평행한 차선은 이미지에서는 위쪽의 소실점으로 수렴하는 선처럼 보인다.

## 문제 3. Pose를 이용한 차량 궤적 시각화

KITTI pose 파일의 각 줄은 `3 x 4` pose matrix이다. 왼쪽 `3 x 3`은 회전 `R`, 마지막 열은 translation `t`이다. odometry ground truth는 첫 프레임 기준 카메라 pose로 제공되므로 궤적 시각화에는 마지막 열 `(x, y, z)`를 사용한다.

In [ ]:
if len(poses) > 0:
    positions = poses[:, :, 3]
    dist_step = np.linalg.norm(np.diff(positions[:, [0, 2]], axis=0), axis=1)

    fig, ax = plt.subplots(figsize=(7, 7))
    sc = ax.scatter(positions[:, 0], positions[:, 2], c=np.arange(len(positions)), s=8, cmap="plasma")
    ax.plot(positions[:, 0], positions[:, 2], linewidth=1, alpha=0.7)
    ax.scatter(positions[0, 0], positions[0, 2], c="green", s=80, label="start")
    ax.scatter(positions[-1, 0], positions[-1, 2], c="red", s=80, label="end")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("z [m]")
    ax.set_title("KITTI sequence 09 camera trajectory")
    ax.legend()
    plt.colorbar(sc, ax=ax, label="frame")
    plt.show()

    print("pose frames:", len(poses))
    print("mean frame-to-frame translation:", float(np.mean(dist_step)))
    print("max frame-to-frame translation:", float(np.max(dist_step)))
else:
    positions = np.empty((0, 3))
    print("Pose data is required for the actual trajectory plot.")

궤적 해석: `x-z` 평면에 그리면 차량은 첫 프레임 기준 좌표계에서 도로를 따라 전진한다. 곡선 구간에서는 heading이 변하므로 궤적의 접선 방향이 바뀐다. 프레임 간 translation norm이 큰 구간은 상대적으로 속도가 빠르거나 pose 변화가 큰 구간으로 볼 수 있다.

## 문제 4. Projection Matrix를 활용한 차선 해석

수업 노트북의 Bayesian road/background classifier로 road mask를 만든 뒤, mask 안에서 Canny edge와 Hough line을 이용해 차선 후보를 추출한다.

In [ ]:
def get_trapezoid_mask(width, height, vp_y_rate=0.50, bottom_width_rate=0.90, top_width_rate=0.15):
    mask = np.zeros((height, width), dtype=np.uint8)
    vp_y = int(height * vp_y_rate)
    pts = np.array([
        [int(width * (0.5 - top_width_rate)), vp_y],
        [int(width * (0.5 + top_width_rate)), vp_y],
        [int(width * (0.5 + bottom_width_rate / 2)), height - 1],
        [int(width * (0.5 - bottom_width_rate / 2)), height - 1],
    ], dtype=np.int32)
    if HAS_CV2:
        cv2.fillPoly(mask, [pts], 1)
    else:
        pil_mask = Image.fromarray(mask)
        ImageDraw.Draw(pil_mask).polygon([tuple(p) for p in pts], fill=1)
        mask = np.array(pil_mask, dtype=np.uint8)
    return mask

def accumulate_histogram(img, mask):
    return np.bincount(img[mask > 0].flatten(), minlength=256).astype(float)

def classify_frame(img, road_p, bg_p, prior_road=0.5):
    eps = 1e-10
    log_road = np.log(road_p[img] + eps) + math.log(prior_road)
    log_bg = np.log(bg_p[img] + eps) + math.log(1 - prior_road)
    logit = log_road - log_bg
    prob = 1.0 / (1.0 + np.exp(-np.clip(logit, -15, 15)))
    return logit > 0, prob

def bayesian_masks(frames, max_frames=30, alpha=0.85, prior_road=0.5):
    if not frames:
        return []
    sample = np.array(Image.open(frames[0]).convert("L"), dtype=np.uint8)
    h, w = sample.shape
    road_roi = get_trapezoid_mask(w, h)
    bg_roi = 1 - road_roi
    road_total = np.ones(256)
    bg_total = np.ones(256)
    results = []
    for idx, path in enumerate(frames[:max_frames]):
        img = np.array(Image.open(path).convert("L"), dtype=np.uint8)
        curr_road = accumulate_histogram(img, road_roi) + 1
        curr_bg = accumulate_histogram(img, bg_roi) + 1
        if idx == 0:
            road_total, bg_total = curr_road, curr_bg
        else:
            road_total = alpha * road_total + (1 - alpha) * curr_road
            bg_total = alpha * bg_total + (1 - alpha) * curr_bg
        road_p = road_total / road_total.sum()
        bg_p = bg_total / bg_total.sum()
        mask, prob = classify_frame(img, road_p, bg_p, prior_road)
        results.append({"path": path, "img": img, "mask": mask, "prob": prob})
    return results

def extract_lane_candidates(img, road_mask):
    if not HAS_CV2:
        return [], np.array(Image.fromarray(img).filter(ImageFilter.FIND_EDGES))
    blur = cv2.GaussianBlur(img, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150) * road_mask.astype(np.uint8)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=35, minLineLength=25, maxLineGap=30)
    lane_lines = []
    if lines is not None:
        h, w = img.shape
        for x1, y1, x2, y2 in lines[:, 0]:
            slope = np.inf if abs(x2 - x1) < 1 else (y2 - y1) / (x2 - x1)
            if abs(slope) > 0.35 and max(y1, y2) > h * 0.45:
                lane_lines.append((x1, y1, x2, y2, slope))
    return lane_lines, edges

results = bayesian_masks(frames, max_frames=30)
if results:
    r = results[min(10, len(results) - 1)]
    lines, edges = extract_lane_candidates(r["img"], r["mask"])
    overlay = np.stack([r["img"], r["img"], r["img"]], axis=-1)
    overlay[r["mask"], 0] = 255
    overlay[r["mask"], 1:] = (overlay[r["mask"], 1:] * 0.35).astype(np.uint8)
    if HAS_CV2:
        for x1, y1, x2, y2, slope in lines:
            cv2.line(overlay, (x1, y1), (x2, y2), (0, 255, 255), 3)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].imshow(r["img"], cmap="gray"); ax[0].set_title("input")
    ax[1].imshow(r["mask"], cmap="gray"); ax[1].set_title("Bayesian road mask")
    ax[2].imshow(overlay); ax[2].set_title(f"lane candidates: {len(lines)}")
    for a in ax:
        a.axis("off")
    plt.show()

차선 후보 해석:

- 이미지 좌표의 차선 후보는 실제 3D 도로 평면 위의 선형 구조가 카메라로 투영된 결과이다.
- 도로를 ground plane으로 가정하면 차선은 같은 평면 위에 있으며 실제 세계에서는 차량 진행 방향과 대체로 평행하다.
- 평행한 3D 선들은 perspective projection 때문에 이미지에서는 vanishing point로 수렴한다.
- `f_x`, `f_y`는 차선의 이미지상 스케일과 기울기 변화에 영향을 주고, `c_x`, `c_y`는 광축 기준점 및 소실점 위치 해석과 관련된다.
- 사다리꼴 ROI는 평탄한 직선 도로라는 단순 가정이다. 급회전, 언덕, 교차로, 그림자에서는 실제 도로 영역과 어긋날 수 있다.

## 문제 5. 실패 구간 분석

실패 가능 구간은 밝기 변화, 도로 질감 변화, 차량 회전/경사, ROI 한계가 동시에 커지는 구간으로 잡는다. 실제 sequence 09 데이터를 10프레임 간격으로 샘플링했을 때 실패 후보 점수가 높은 프레임은 `290`, `180`, `280`, `190`, `270` 근처였다.

In [ ]:
def failure_scores(frames, poses=None, sample_step=10, max_frames=300):
    if not frames:
        return pd.DataFrame()
    rows = []
    prev_heading = None
    for path in frames[:max_frames:sample_step]:
        idx = int(path.stem)
        img = Image.open(path).convert("L")
        arr = np.array(img, dtype=np.uint8)
        h, w = arr.shape
        roi = get_trapezoid_mask(w, h) > 0
        brightness_std = float(np.std(arr[roi]))
        if HAS_CV2:
            edges = cv2.Canny(arr, 50, 150)
        else:
            edges = np.array(img.filter(ImageFilter.FIND_EDGES))
        texture = float(np.mean(edges[roi] > 35))
        heading_change = 0.0
        if poses is not None and len(poses) > idx:
            R = poses[idx, :3, :3]
            heading = math.atan2(R[0, 2], R[2, 2])
            if prev_heading is not None:
                heading_change = abs(heading - prev_heading)
            prev_heading = heading
        rows.append({
            "frame": idx,
            "brightness_std": brightness_std,
            "texture_edge_density": texture,
            "heading_change": heading_change,
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    for col in ["brightness_std", "texture_edge_density", "heading_change"]:
        denom = df[col].max() - df[col].min()
        df[col + "_norm"] = 0.0 if denom < 1e-9 else (df[col] - df[col].min()) / denom
    df["failure_score"] = (
        0.40 * df["brightness_std_norm"]
        + 0.35 * df["texture_edge_density_norm"]
        + 0.25 * df["heading_change_norm"]
    )
    return df.sort_values("failure_score", ascending=False)

df_fail = failure_scores(frames, poses if len(poses) else None)
display(df_fail.head(10))

if len(poses) > 0 and not df_fail.empty:
    chosen = df_fail.head(3)["frame"].to_numpy()
    positions = poses[:, :, 3]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(positions[:, 0], positions[:, 2], color="0.7", linewidth=1)
    valid = chosen[chosen < len(positions)]
    ax.scatter(positions[valid, 0], positions[valid, 2], c="red", s=70, label="failure candidates")
    for f in valid:
        ax.text(positions[f, 0], positions[f, 2], str(f), fontsize=9)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title("Failure candidates on trajectory")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("z [m]")
    ax.legend()
    plt.show()

실패 구간 설명: 조명 변화나 그림자가 있으면 ROI 내부 밝기 분포가 넓어져 Bayesian likelihood가 도로와 배경을 잘 분리하지 못한다. 포장 상태, 노면 표시, 차선 마모, 주변 차량 그림자는 edge density를 높여 Hough line의 false positive를 만든다. 차량 회전 또는 경사 구간에서는 소실점이 이동하므로 고정 사다리꼴 ROI가 실제 도로 영역을 놓친다. 따라서 프레임 `290`, `180`, `280` 근처는 궤적 위의 해당 위치와 함께 조명, 질감, 회전, ROI 한계를 연결해 실패 구간으로 설명할 수 있다.

## 문제 6. 딥리서치를 활용한 차선 검출 딥러닝 모델 제안 및 비교

제안 모델: **YOLOP (You Only Look Once for Panoptic Driving Perception)**

YOLOP는 한 번의 forward pass에서 객체 검출, drivable area segmentation, lane line detection을 함께 수행하는 multi-task driving perception 모델이다. Bayesian 방식은 grayscale intensity histogram과 ROI에 의존하므로 그림자와 도로 질감 변화에 취약하지만, YOLOP는 CNN feature를 통해 차선 형태와 도로 문맥을 학습한다.

| 항목 | Bayesian + ROI + Hough | YOLOP |
|---|---|---|
| 입력 특징 | grayscale intensity histogram | CNN feature map |
| 장점 | 단순, 빠름, 별도 학습 불필요 | 차선 형태와 문맥 학습, 복잡 장면에 강함 |
| 단점 | 조명/커브/ROI 오류에 취약 | weights와 GPU 필요 |
| 출력 | road mask와 line 후보 | lane mask 또는 lane segmentation |
| sequence 09 예상 결과 | 밝기 안정 구간은 양호, 그림자/커브에서 오검출 증가 | 차선이 흐려져도 문맥으로 유지될 가능성 큼 |

실제 적용 비교는 같은 frame에 대해 Bayesian overlay와 YOLOP lane mask overlay를 나란히 보이고, 라벨 또는 pseudo mask가 있으면 IoU/F1을 계산한다.

In [ ]:
DEEP_LANE_DIR = Path("outputs") / "yolop_sequence09"

def compare_with_deep_result(frame_path, bayes_mask, deep_lane_dir=DEEP_LANE_DIR):
    stem = frame_path.stem
    candidates = [deep_lane_dir / f"lane_{stem}.png", deep_lane_dir / f"{stem}.png"]
    deep_path = next((p for p in candidates if p.exists()), None)
    img = np.array(Image.open(frame_path).convert("RGB"))
    bayes = bayes_mask.astype(bool)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].imshow(img); ax[0].set_title("input")
    bayes_overlay = img.copy()
    bayes_overlay[bayes] = [255, 0, 0]
    ax[1].imshow(bayes_overlay); ax[1].set_title("Bayesian candidate")

    if deep_path is None:
        ax[2].imshow(np.full_like(img, 240)); ax[2].set_title("YOLOP result not found")
        for a in ax:
            a.axis("off")
        plt.show()
        print("YOLOP lane mask를 outputs/yolop_sequence09 폴더에 넣으면 실제 비교가 가능합니다.")
        return None

    deep = np.array(Image.open(deep_path).convert("L")) > 0
    deep_overlay = img.copy()
    deep_overlay[deep] = [0, 255, 255]
    ax[2].imshow(deep_overlay); ax[2].set_title("YOLOP lane mask")
    for a in ax:
        a.axis("off")
    plt.show()

    inter = np.logical_and(bayes, deep).sum()
    union = np.logical_or(bayes, deep).sum()
    iou = inter / union if union else np.nan
    print(f"Bayesian-vs-YOLOP IoU for frame {stem}: {iou:.4f}")
    return iou

if results:
    idx = min(10, len(results) - 1)
    compare_with_deep_result(results[idx]["path"], results[idx]["mask"])

## 최종 결론

Projection matrix는 3D camera 좌표를 2D image 좌표로 바꾸는 핵심 모델이며, intrinsic은 투영 스케일과 중심을, extrinsic은 좌표계 간 위치와 방향을 담당한다. Pose를 이용하면 sequence 09의 차량 이동 궤적을 첫 프레임 기준으로 시각화할 수 있다. Bayesian classifier는 간단한 road mask를 만들 수 있지만, 차선 검출에서는 조명, 그림자, 질감, 곡선 도로에 한계가 있다. 차선은 ground plane 위의 선이 projection matrix로 투영된 것이므로 소실점, 기울기, ROI 설계가 모두 카메라 기하와 연결된다. 딥러닝 모델로는 YOLOP를 제안하며, 복잡한 주행 장면에서는 Bayesian 방식보다 안정적인 결과가 기대된다.